# Imagenette local training

Download [Imagenette](https://github.com/fastai/imagenette) from the internet with torchvision (`download=True`) and train a small CNN locally (CPU or GPU).

Imagenette is a 10-class ImageNet subset (tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute). This notebook uses the 320px archive (~325 MB).

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import Compose, Resize, ToTensor

In [2]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, inputs):
        features = self.features(inputs)
        features = torch.flatten(features, 1)
        return self.classifier(features)

In [3]:
def get_dataset():
    return datasets.Imagenette(
        root="/tmp/data",
        split="train",
        size="320px",
        download=True,
        transform=Compose([
            Resize((64, 64)),
            ToTensor(),
        ]),
    )

In [4]:
def train():
    num_epochs = 3
    batch_size = 64

    dataset = get_dataset()
    print(f"downloaded samples: {len(dataset)}")
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = NeuralNetwork().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    for epoch in range(num_epochs):
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            pred = model(inputs)
            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()
        print(f"epoch: {epoch}, loss: {loss.item()}")

In [ ]:
train()

downloaded samples: 9469
epoch: 0, loss: 2.2973978519439697
